# NSL-KDD Intrusion Detection

**Objective:** Train and evaluate ML models (Logistic Regression, Random Forest) to detect network attacks using NSL-KDD dataset.

### Import Packages

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix

from src.preprocess import load_data, split_and_preprocess

### Load and preprocess the data for training

In [ ]:
df_train, df_test = load_data('../data/KDDTrain+.txt', '../data/KDDTest+.txt')
X_train, X_test, y_train, y_test = split_and_preprocess(df_train, df_test)

#### Logistic Regression Training

In [ ]:
params = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    LogisticRegression(max_iter=1000),
    params,
    scoring='f1'
)

grid.fit(X_train, y_train)
lr = grid.best_estimator_

print(f'Best params: {grid.best_params_}')

#### Random Forest Training

In [ ]:
rf = RandomForestClassifier(
    n_estimators=400,
    max_features='sqrt',
    n_jobs=-1
)

rf.fit(X_train, y_train)

#### Save the models

In [ ]:
joblib.dump(lr, '../models/logistic_regression.joblib')
joblib.dump(rf, '../models/random_forest.joblib')

You can also load the saved models instead of training them (training takes ~20 seconds)

In [ ]:
lr = joblib.load('../models/logistic_regression.joblib')
rf = joblib.load('../models/random_forest.joblib')

### Logistic Regression

In [ ]:
threshold = 0.3

y_proba = lr.predict_proba(X_test)[:,1]
y_pred = (y_proba > threshold).astype(int)

report = classification_report(y_test, y_pred)
conf_mat = confusion_matrix(y_test, y_pred)

print(f'Classification report for Logistic Regression (threshold: {threshold}):')
print(report)

sns.heatmap(conf_mat, cmap='Blues')
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Random Forest Classifier

In [ ]:
threshold = 0.3
y_proba = rf.predict_proba(X_test)[:,1]
y_pred = (y_proba > threshold).astype(int)

report = classification_report(y_test, y_pred)
conf_mat = confusion_matrix(y_test, y_pred)

print(f'Classification report for Random Forest Classifier:')
print(report)

sns.heatmap(conf_mat, cmap='Blues')
plt.title('Random Forest Classifier Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()